In [1]:
# Install libraries
!pip install pandas numpy scikit-learn imbalanced-learn

print("✅ Done!")

✅ Done!


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [3]:
# Column names
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes',
    'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Load data
train_df = pd.read_csv(
    'https://raw.githubusercontent.com/logesh-GIT001/Smart-SOC/main/data/raw/KDDTrain%2B.txt',
    names=columns
)

test_df = pd.read_csv(
    'https://raw.githubusercontent.com/logesh-GIT001/Smart-SOC/main/data/raw/KDDTest%2B.txt',
    names=columns
)

print(f"✅ Train: {train_df.shape}")
print(f"✅ Test:  {test_df.shape}")

✅ Train: (125973, 43)
✅ Test:  (22544, 43)


In [4]:
# Drop useless columns we found in EDA
# num_outbound_cmds → always 0, useless
# difficulty → not a network feature, just a dataset label

drop_cols = ['num_outbound_cmds', 'difficulty']

train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

print(f"✅ Train shape after drop: {train_df.shape}")
print(f"✅ Test shape after drop:  {test_df.shape}")
print(f"✅ Dropped columns: {drop_cols}")

✅ Train shape after drop: (125973, 41)
✅ Test shape after drop:  (22544, 41)
✅ Dropped columns: ['num_outbound_cmds', 'difficulty']


In [5]:
# Convert text columns to numbers
# protocol_type, service, flag → ML models only understand numbers

le = LabelEncoder()

cat_cols = ['protocol_type', 'service', 'flag']

for col in cat_cols:
    # Fit on both train and test together so all values are covered
    all_values = pd.concat([train_df[col], test_df[col]])
    le.fit(all_values)
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

print("✅ Categorical columns encoded!")
print(f"\nSample encoded values:")
print(train_df[['protocol_type', 'service', 'flag']].head())

✅ Categorical columns encoded!

Sample encoded values:
   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9


In [6]:
# Convert attack categories to numbers
# Normal, DoS, Probe, R2L, U2R → 0, 1, 2, 3, 4

def categorize(label):
    dos   = ['neptune','back','land','pod','smurf','teardrop']
    probe = ['ipsweep','nmap','portsweep','satan']
    r2l   = ['ftp_write','guess_passwd','imap','multihop',
             'phf','spy','warezclient','warezmaster']
    u2r   = ['buffer_overflow','loadmodule','perl','rootkit']
    if label == 'normal': return 0
    elif label in dos:    return 1
    elif label in probe:  return 2
    elif label in r2l:    return 3
    elif label in u2r:    return 4
    else:                 return 1  # unknown → treat as DoS

train_df['label'] = train_df['label'].apply(categorize)
test_df['label']  = test_df['label'].apply(categorize)

print("✅ Labels encoded!")
print("\nLabel distribution:")
print(train_df['label'].value_counts())
print("\n0=Normal, 1=DoS, 2=Probe, 3=R2L, 4=U2R")

✅ Labels encoded!

Label distribution:
label
0    67343
1    45927
2    11656
3      995
4       52
Name: count, dtype: int64

0=Normal, 1=DoS, 2=Probe, 3=R2L, 4=U2R


In [7]:
# Scale all features so large values don't dominate
# src_bytes max = 1.3 billion vs land max = 1 → unfair to model!

X_train = train_df.drop(columns=['label'])
y_train = train_df['label']

X_test = test_df.drop(columns=['label'])
y_test = test_df['label']

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Scaling done!")
print(f"\nX_train shape: {X_train_scaled.shape}")
print(f"X_test shape:  {X_test_scaled.shape}")
print(f"\nBefore scaling - src_bytes max: {X_train['src_bytes'].max()}")
print(f"After scaling  - src_bytes max: {X_train_scaled[:, 4].max():.4f}")

✅ Scaling done!

X_train shape: (125973, 40)
X_test shape:  (22544, 40)

Before scaling - src_bytes max: 1379963888
After scaling  - src_bytes max: 235.0675


In [8]:
# SMOTE creates synthetic samples for rare classes
# U2R only has 52 records → model will barely learn it!

print("Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c}")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c}")

print(f"\n✅ Balanced dataset shape: {X_train_balanced.shape}")

Before SMOTE:
  Class 0: 67343
  Class 1: 45927
  Class 2: 11656
  Class 3: 995
  Class 4: 52

After SMOTE:
  Class 0: 67343
  Class 1: 67343
  Class 2: 67343
  Class 3: 67343
  Class 4: 67343

✅ Balanced dataset shape: (336715, 40)


In [9]:
import os

# Create processed folder
os.makedirs('processed', exist_ok=True)

# Save everything
np.save('processed/X_train.npy', X_train_balanced)
np.save('processed/y_train.npy', y_train_balanced)
np.save('processed/X_test.npy',  X_test_scaled)
np.save('processed/y_test.npy',  y_test.values)

# Save feature names
feature_names = list(X_train.columns)
pd.Series(feature_names).to_csv('processed/feature_names.csv', index=False)

print("✅ All files saved!")
print("\nSaved files:")
for f in os.listdir('processed'):
    size = os.path.getsize(f'processed/{f}')
    print(f"  {f} → {size/1024:.1f} KB")

✅ All files saved!

Saved files:
  X_test.npy → 7045.1 KB
  X_train.npy → 105223.6 KB
  feature_names.csv → 0.6 KB
  y_test.npy → 176.2 KB
  y_train.npy → 2630.7 KB


In [10]:
print("=" * 45)
print("   PREPROCESSING COMPLETE!")
print("=" * 45)
print(f"\n✅ Dropped useless features  : 2")
print(f"✅ Encoded categorical cols  : 3")
print(f"✅ Encoded labels            : 5 classes")
print(f"✅ Scaled all features       : 40 features")
print(f"✅ Balanced with SMOTE       : 336,715 records")
print(f"\nReady for model training! 🚀")
print(f"\nDAY 4 → Train Random Forest + XGBoost")

   PREPROCESSING COMPLETE!

✅ Dropped useless features  : 2
✅ Encoded categorical cols  : 3
✅ Encoded labels            : 5 classes
✅ Scaled all features       : 40 features
✅ Balanced with SMOTE       : 336,715 records

Ready for model training! 🚀

DAY 4 → Train Random Forest + XGBoost
